# 85 — Skema 2 Cross-Dataset 3-class → Primer 3-class

**Motivasi:** train di benchmark dataset (CK+/JAFFE/RAF-DB/KDEF) dengan label 3-class, lalu test di Primer 3-class test set. **Inference only** — pakai checkpoint dari nb 84.

**Scope:** 4 sources × 7 archs (single + 2 Late Fusion variants) = **36 configs**.

**Estimasi:** ~30 menit di T4 (inference saja, no training).

**Prerequisite:** nb 84 selesai dulu di VPS untuk generate checkpoint per dataset.


In [1]:
import sys, os, json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
    EmotionEarlyFusion, EmotionEarlyFusionTransfer,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
BENCH_BASE = PROJECT_ROOT / 'models' / 'benchmark'
NUM_CLASSES = 3
EMOTIONS = ['positive', 'neutral', 'negative']
REMAP_3 = np.array([1, 0, 2, 2, 2, 2, 0], dtype=np.int64)
BATCH = 32

# Load Primer test (3-class)
img_te = np.load(PRIMER_DIR / 'X_test_images.npy').astype(np.float32)
lm_te  = np.load(PRIMER_DIR / 'X_test_landmarks.npy').astype(np.float32)
hm_te  = np.load(PRIMER_DIR / 'X_test_heatmaps.npy').astype(np.float32)
y_te   = REMAP_3[np.load(PRIMER_DIR / 'y_test.npy')]
print(f'Primer test: {len(y_te)} samples, dist {np.bincount(y_te, minlength=3).tolist()}')


Primer test: 929 samples, dist [186, 688, 55]


In [2]:
# ── Helpers ──
def stack_4ch(img, hm):
    if hm.ndim == 3: hm = hm[..., None]
    return np.concatenate([img, hm], axis=-1).astype(np.float32)

def load_te_loader(arch):
    y_t = torch.from_numpy(y_te).long()
    if arch == 'fcnn':
        return DataLoader(TensorDataset(torch.from_numpy(lm_te).float(), y_t),
                          batch_size=BATCH, num_workers=0)
    if arch == 'cnn':
        t = torch.from_numpy(img_te).permute(0, 3, 1, 2).float()
        return DataLoader(TensorDataset(t, y_t), batch_size=BATCH, num_workers=0)
    if arch == 'fusion':
        t = torch.from_numpy(img_te).permute(0, 3, 1, 2).float()
        return DataLoader(TensorDataset(t, torch.from_numpy(lm_te).float(), y_t),
                          batch_size=BATCH, num_workers=0)
    if arch == 'earlyfusion':
        x4 = stack_4ch(img_te, hm_te)
        t = torch.from_numpy(x4).permute(0, 3, 1, 2).float()
        return DataLoader(TensorDataset(t, y_t), batch_size=BATCH, num_workers=0)


def predict_arch(model, arch, loader):
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for batch in loader:
            *x, y = [b.to(device) for b in batch]
            out = model(*x) if arch == 'fusion' else model(x[0])
            yt.append(y.cpu().numpy()); yp.append(out.argmax(1).cpu().numpy())
    return np.concatenate(yt), np.concatenate(yp)


def softmax_arch(model, arch, loader):
    model.eval()
    p = []
    with torch.no_grad():
        for batch in loader:
            *x, _ = [b.to(device) for b in batch]
            out = model(*x) if arch == 'fusion' else model(x[0])
            p.append(F.softmax(out, dim=1).cpu().numpy())
    return np.concatenate(p)


def metrics_pack(yt, yp):
    return {
        'test_macro_f1':    float(f1_score(yt, yp, average='macro', zero_division=0)),
        'test_micro_f1':    float(f1_score(yt, yp, average='micro', zero_division=0)),
        'test_weighted_f1': float(f1_score(yt, yp, average='weighted', zero_division=0)),
        'test_accuracy':    float(accuracy_score(yt, yp)),
        'confusion_matrix': confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES))).tolist(),
        'classification_report': classification_report(yt, yp, target_names=EMOTIONS,
                                                       labels=list(range(NUM_CLASSES)),
                                                       zero_division=0, output_dict=True),
    }


In [3]:
# ── Cross-dataset inference loop ──
ARCH_BUILDERS = {
    'CNN':              (lambda: EmotionCNN(num_classes=NUM_CLASSES),                 'cnn'),
    'FCNN':             (lambda: EmotionFCNN(num_classes=NUM_CLASSES),                'fcnn'),
    'Intermediate':     (lambda: IntermediateFusion(num_classes=NUM_CLASSES),         'fusion'),
    'CNN_TL':           (lambda: EmotionCNNTransfer(num_classes=NUM_CLASSES),         'cnn'),
    'Intermediate_TL':  (lambda: IntermediateFusionTransfer(num_classes=NUM_CLASSES), 'fusion'),
    'EarlyFusion':      (lambda: EmotionEarlyFusion(num_classes=NUM_CLASSES),         'earlyfusion'),
    'EarlyFusion_TL':   (lambda: EmotionEarlyFusionTransfer(num_classes=NUM_CLASSES), 'earlyfusion'),
}

cross_results = {}
for ds in ['ckplus', 'jaffe', 'rafdb', 'kdef']:
    bench_dir = BENCH_BASE / ds / '3class'
    print(f"\n{'='*70}\n  {ds.upper()} → Primer (3-class)\n{'='*70}")

    for arch_name, (build_fn, arch_type) in ARCH_BUILDERS.items():
        ckpt = bench_dir / f'{arch_name}_b1.pth'
        if not ckpt.exists():
            print(f'  [SKIP] {arch_name}: {ckpt} missing')
            continue
        model = build_fn().to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        loader = load_te_loader(arch_type)
        yt, yp = predict_arch(model, arch_type, loader)
        m = metrics_pack(yt, yp)
        cfg = f'{ds}_to_primer_{arch_name}'
        cross_results[cfg] = m
        print(f'  {arch_name:<20}: macro={m["test_macro_f1"]:.4f}  acc={m["test_accuracy"]:.4f}')

    # Late Fusion (2 variants)
    # Dedicated checkpoints (LateFusion_CNN.pth) dipakai jika ada (ckplus/jaffe dari laptop).
    # Fallback ke CNN_b1.pth / CNN_TL_b1.pth jika tidak ada (rafdb/kdef dari GPU lab).
    for variant, cnn_cls, dedicated_cnn, fallback_cnn, dedicated_fcnn, fallback_fcnn, json_key in [
        ('Late_Fusion',    EmotionCNN,
         'LateFusion_CNN.pth',    'CNN_b1.pth',
         'LateFusion_FCNN.pth',   'FCNN_b1.pth',
         'LateFusion'),
        ('Late_Fusion_TL', EmotionCNNTransfer,
         'LateFusion_TL_CNN.pth', 'CNN_TL_b1.pth',
         'LateFusion_TL_FCNN.pth','FCNN_b1.pth',
         'LateFusion_TL'),
    ]:
        cnn_ckpt  = bench_dir / (dedicated_cnn  if (bench_dir / dedicated_cnn).exists()  else fallback_cnn)
        fcnn_ckpt = bench_dir / (dedicated_fcnn if (bench_dir / dedicated_fcnn).exists() else fallback_fcnn)
        if not (cnn_ckpt.exists() and fcnn_ckpt.exists()):
            print(f'  [SKIP] {variant}: checkpoints missing')
            continue
        cnn = cnn_cls(num_classes=NUM_CLASSES).to(device)
        cnn.load_state_dict(torch.load(cnn_ckpt, map_location=device, weights_only=True))
        fcnn = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
        fcnn.load_state_dict(torch.load(fcnn_ckpt, map_location=device, weights_only=True))

        # Ambil weight_cnn dari run terbaik (val_f1 tertinggi) di results.json
        bench_results = json.load(open(bench_dir / f'{ds}_3c_results.json'))
        runs = bench_results.get(json_key, {}).get('runs', [])
        best_run = max(runs, key=lambda r: r['val_f1']) if runs else {}
        best_w = best_run.get('weight_cnn', 0.5)

        p_cnn  = softmax_arch(cnn, 'cnn', load_te_loader('cnn'))
        p_fcnn = softmax_arch(fcnn, 'fcnn', load_te_loader('fcnn'))
        fused = best_w * p_cnn + (1 - best_w) * p_fcnn
        pred = fused.argmax(1)
        m = metrics_pack(y_te, pred)
        m['best_cnn_weight'] = best_w
        cfg = f'{ds}_to_primer_{variant}'
        cross_results[cfg] = m
        print(f'  {variant:<20}: w={best_w:.2f} macro={m["test_macro_f1"]:.4f} acc={m["test_accuracy"]:.4f}')

with open(BENCH_BASE / 'all_3c_skema2_cross_results.json', 'w') as f:
    json.dump(cross_results, f, indent=2)
print(f"\nSaved master: {BENCH_BASE / 'all_3c_skema2_cross_results.json'}")



  CKPLUS → Primer (3-class)


  CNN                 : macro=0.3690  acc=0.6997


  FCNN                : macro=0.3223  acc=0.3003


  Intermediate        : macro=0.2731  acc=0.3628


  CNN_TL              : macro=0.2795  acc=0.4898


...

  Intermediate_TL     : macro=0.2072  acc=0.1873


  EarlyFusion         : macro=0.0868  acc=0.1012


  EarlyFusion_TL      : macro=0.0925  acc=0.1066


  Late_Fusion         : w=0.60 macro=0.1142 acc=0.1206


  Late_Fusion_TL      : w=0.55 macro=0.2179 acc=0.1970

Saved master: /mnt/extended-home/fitra_dosen/2025_iris_fer_taufik/MultimodalEmoLearn/models/benchmark/all_3c_skema2_cross_results.json


## Commit

```bash
git add models/benchmark/all_3c_skema2_cross_results.json notebooks/results/85_*
git commit -m "Add 3-class Skema 2 cross-dataset → Primer (nb 85, inference)"
```
